# Staffing Plan Employee Generation

This notebook builds a realistic, uneven staffing plan for USPS-style Texas facilities, generates staging-ready employee rows, and exports CSV files for later MySQL loading. It does not connect to or modify MySQL.

## Setup

Run this notebook from the project root or from this `Employee` folder. If `faker` is missing, install it once in the active notebook kernel with `%pip install faker`.

In [1]:
from __future__ import annotations

import csv
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from faker import Faker
except ImportError as exc:
    raise ImportError("Install faker in this notebook kernel with: %pip install faker") from exc

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

## Configuration

In [2]:
TOTAL_TARGET_EMPLOYEES = 5000
RANDOM_SEED = 42
OUTPUT_DIR = "output"

NOTEBOOK_DIR = Path.cwd()
if NOTEBOOK_DIR.name.lower() != "employee":
    NOTEBOOK_DIR = Path("Synthetic MySQL Tables") / "Employee"

PROJECT_ROOT = NOTEBOOK_DIR.parents[1]

FACILITIES_PATH = PROJECT_ROOT / "Synthetic MySQL Tables" / "Facilities" / "postal_tx_facilities.csv"
ZIP_CENTROIDS_PATH = PROJECT_ROOT / "Texas_ZIP_Geo" / "texas_zip_centroids.csv"
DEPARTMENTS_PATH = PROJECT_ROOT / "Synthetic MySQL Tables" / "Departments" / "departments.csv"
OUTPUT_PATH = NOTEBOOK_DIR / OUTPUT_DIR

# Used only to recover facility_id values when the facilities CSV does not include them.
CURRENT_SCHEMA_FACILITY_SQL = Path(r"C:\Users\Ryan\OneDrive\Documents\dumps\CurrentSchema\postal_bi_system_facility.sql")

EMAIL_DOMAIN = "postal-demo.local"
GENERATED_SOURCE = "staffing_plan_employee_generation.ipynb"

FACILITY_TYPE_NAMES = {
    1: "Post Office",
    2: "Vehicle Maintenance",
    3: "Administrative Office",
    4: "Network Facilities",
    5: "Mail Processing",
}

DEPARTMENT_TYPE_NAMES = {
    1: "Retail Services",
    2: "Delivery",
    3: "Logistics",
    4: "Operations",
    5: "Maintenance",
    6: "Administrative",
}

FACILITY_TYPE_MULTIPLIERS = {
    "Post Office": 1.00,
    "Vehicle Maintenance": 0.70,
    "Administrative Office": 0.60,
    "Network Facilities": 1.80,
    "Mail Processing": 2.40,
}

FACILITY_MINIMUMS = {
    "Post Office": 4,
    "Vehicle Maintenance": 3,
    "Administrative Office": 3,
    "Network Facilities": 10,
    "Mail Processing": 15,
}

DEPARTMENT_SHARES = {
    "Post Office": {"Delivery": 0.55, "Retail Services": 0.25, "Logistics": 0.20},
    "Vehicle Maintenance": {"Maintenance": 1.00},
    "Administrative Office": {"Administrative": 1.00},
    "Network Facilities": {"Logistics": 0.35, "Operations": 0.30, "Delivery": 0.25, "Administrative": 0.10},
    "Mail Processing": {"Operations": 0.45, "Logistics": 0.30, "Maintenance": 0.15, "Administrative": 0.10},
}

FACILITY_MANAGER_PRIORITY = {
    "Post Office": ["Delivery", "Retail Services", "Logistics"],
    "Vehicle Maintenance": ["Maintenance"],
    "Administrative Office": ["Administrative"],
    "Network Facilities": ["Operations", "Logistics", "Delivery", "Administrative"],
    "Mail Processing": ["Operations", "Logistics", "Maintenance", "Administrative"],
}

REGULAR_SALARY_BANDS = {
    "Retail Services": (38000, 52000),
    "Delivery": (42000, 62000),
    "Logistics": (40000, 60000),
    "Operations": (45000, 68000),
    "Maintenance": (46000, 70000),
    "Administrative": (40000, 58000),
}

MANAGER_SALARY_BANDS = {
    "Retail Services": (75000, 95000),
    "Delivery": (78000, 100000),
    "Logistics": (78000, 102000),
    "Operations": (85000, 110000),
    "Maintenance": (80000, 105000),
    "Administrative": (75000, 98000),
}

JOB_TITLES = {
    "Retail Services": ["Retail Clerk", "Window Clerk", "Retail Associate", "Customer Service Clerk"],
    "Delivery": ["City Carrier", "Rural Carrier", "Delivery Associate", "Mail Carrier"],
    "Logistics": ["Mail Handler", "Routing Clerk", "Logistics Associate", "Transportation Clerk"],
    "Operations": ["Processing Clerk", "Sortation Worker", "Operations Associate", "Plant Operations Clerk"],
    "Maintenance": ["Maintenance Technician", "Vehicle Technician", "Equipment Maintenance Technician", "Facility Maintenance Associate"],
    "Administrative": ["Administrative Assistant", "Office Coordinator", "Records Clerk", "Administrative Specialist"],
}

MANAGER_TITLES = {
    "Retail Services": "Retail Services Manager",
    "Delivery": "Delivery Manager",
    "Logistics": "Logistics Manager",
    "Operations": "Operations Manager",
    "Maintenance": "Maintenance Manager",
    "Administrative": "Administrative Manager",
}

## Helper Functions

In [3]:
def read_csv_auto(path: Path) -> pd.DataFrame:
    """Read comma or semicolon CSV exports without mutating the source file."""
    if not path.exists():
        raise FileNotFoundError(f"Required input file was not found: {path}")
    return pd.read_csv(path, sep=None, engine="python")


def clean_zip_codes(frame: pd.DataFrame, column: str = "zip_code") -> pd.DataFrame:
    cleaned = frame.copy()
    if column not in cleaned.columns and "zip" in cleaned.columns:
        cleaned = cleaned.rename(columns={"zip": column})
    cleaned[column] = (
        cleaned[column]
        .astype("string")
        .str.extract(r"(\d{5})", expand=False)
        .fillna("")
    )
    return cleaned


def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value).strip().upper())


def slugify(value: object) -> str:
    text = re.sub(r"[^a-zA-Z0-9]+", "", str(value).lower())
    return text or "employee"


def truncate_text(value: str, max_length: int) -> str:
    text = str(value).replace("\n", " ").strip()
    return text[:max_length]


def split_sql_tuples(values_text: str) -> list[str]:
    tuples = []
    depth = 0
    in_quote = False
    current = []
    i = 0
    while i < len(values_text):
        char = values_text[i]
        if char == "'":
            current.append(char)
            if i + 1 < len(values_text) and values_text[i + 1] == "'":
                current.append(values_text[i + 1])
                i += 2
                continue
            in_quote = not in_quote
        elif not in_quote and char == "(":
            depth += 1
            if depth > 1:
                current.append(char)
        elif not in_quote and char == ")":
            depth -= 1
            if depth == 0:
                tuples.append("".join(current))
                current = []
            else:
                current.append(char)
        elif depth > 0:
            current.append(char)
        i += 1
    return tuples


def parse_sql_tuple(tuple_text: str) -> list[object]:
    row = next(csv.reader([tuple_text], delimiter=",", quotechar="'", skipinitialspace=True))
    parsed = []
    for value in row:
        if value.upper() == "NULL":
            parsed.append(np.nan)
        else:
            parsed.append(value)
    return parsed


def load_facility_sql_lookup(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    text = path.read_text(encoding="utf-8", errors="ignore")
    match = re.search(r"INSERT INTO `facility` VALUES\s*(.*);\s*/\*!40000 ALTER TABLE `facility` ENABLE KEYS", text, flags=re.DOTALL)
    if not match:
        return pd.DataFrame()

    columns = [
        "facility_id", "facility_type_id", "manager_employee_id", "facility_name", "street_address",
        "county", "city", "state_code", "zip_code", "facility_department_prefix", "territory_id",
        "created_at", "updated_at",
    ]
    rows = [parse_sql_tuple(item) for item in split_sql_tuples(match.group(1))]
    lookup = pd.DataFrame(rows, columns=columns)
    for column in ["facility_id", "facility_type_id", "territory_id"]:
        lookup[column] = pd.to_numeric(lookup[column], errors="coerce").astype("Int64")
    lookup["facility_type_name"] = lookup["facility_type_id"].map(FACILITY_TYPE_NAMES)
    lookup = clean_zip_codes(lookup, "zip_code")
    lookup["facility_name_key"] = lookup["facility_name"].map(normalize_text)
    return lookup


def load_inputs() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    facilities = read_csv_auto(FACILITIES_PATH)
    zip_geo = read_csv_auto(ZIP_CENTROIDS_PATH)
    departments = read_csv_auto(DEPARTMENTS_PATH)

    facilities = clean_zip_codes(facilities, "zip_code")
    zip_geo = clean_zip_codes(zip_geo, "zip_code")

    departments.columns = [column.strip() for column in departments.columns]
    for column in ["department_id", "facility_id", "department_type_id"]:
        departments[column] = pd.to_numeric(departments[column], errors="coerce").astype("Int64")
    if "department_type_name" not in departments.columns:
        departments["department_type_name"] = departments["department_type_id"].map(DEPARTMENT_TYPE_NAMES)

    facility_lookup = load_facility_sql_lookup(CURRENT_SCHEMA_FACILITY_SQL)
    facilities["facility_name_key"] = facilities["facility_name"].map(normalize_text)

    if "facility_id" not in facilities.columns:
        if facility_lookup.empty:
            raise ValueError(
                "postal_tx_facilities.csv does not include facility_id and the facility SQL lookup was not found. "
                "Export facility_id with the facilities file or update CURRENT_SCHEMA_FACILITY_SQL."
            )
        facilities = facilities.merge(
            facility_lookup[["facility_id", "facility_name_key", "zip_code"]],
            on=["facility_name_key", "zip_code"],
            how="left",
        )
        unmatched_mask = facilities["facility_id"].isna()
        for index, row in facilities.loc[unmatched_mask].iterrows():
            candidates = facility_lookup[
                (facility_lookup["zip_code"] == row["zip_code"])
                & (facility_lookup["facility_type_id"].astype("Int64") == int(row["facility_type_id"]))
            ]
            if len(candidates) == 1:
                facilities.loc[index, "facility_id"] = candidates["facility_id"].iloc[0]

    facilities["facility_id"] = pd.to_numeric(facilities["facility_id"], errors="coerce").astype("Int64")
    if facilities["facility_id"].isna().any():
        missing = facilities.loc[facilities["facility_id"].isna(), ["facility_name", "zip_code"]].head(10)
        raise ValueError(f"Some facilities could not be matched to a facility_id. Sample:\n{missing}")

    if not facility_lookup.empty:
        missing_ids = sorted(set(departments["facility_id"].dropna().astype(int)) - set(facilities["facility_id"].dropna().astype(int)))
        if missing_ids:
            extra = facility_lookup[facility_lookup["facility_id"].astype("Int64").isin(missing_ids)].copy()
            for column in facilities.columns:
                if column not in extra.columns:
                    extra[column] = np.nan
            facilities = pd.concat([facilities, extra[facilities.columns]], ignore_index=True)

    facilities = facilities.drop_duplicates(subset=["facility_id"], keep="first")
    return facilities, zip_geo, departments


def safe_log_median(series: pd.Series) -> float:
    logged = np.log1p(pd.to_numeric(series, errors="coerce"))
    median = logged.replace([np.inf, -np.inf], np.nan).dropna().median()
    return float(median) if pd.notna(median) and median > 0 else 1.0


def calculate_facility_weights(facilities: pd.DataFrame, zip_geo: pd.DataFrame) -> pd.DataFrame:
    facility_data = facilities.copy()
    facility_data["facility_type_name"] = facility_data["facility_type_name"].fillna(
        facility_data["facility_type_id"].map(FACILITY_TYPE_NAMES)
    )

    zip_data = zip_geo[["zip_code", "population", "density"]].copy()
    zip_data["population"] = pd.to_numeric(zip_data["population"], errors="coerce")
    zip_data["density"] = pd.to_numeric(zip_data["density"], errors="coerce")

    facility_data = facility_data.merge(zip_data, on="zip_code", how="left")
    facility_data["interior_sq_ft"] = pd.to_numeric(facility_data["interior_sq_ft"], errors="coerce")
    facility_data["population"] = pd.to_numeric(facility_data["population"], errors="coerce")
    facility_data["density"] = pd.to_numeric(facility_data["density"], errors="coerce")

    facility_data["interior_sq_ft_imputed"] = facility_data["interior_sq_ft"].isna() | (facility_data["interior_sq_ft"] <= 0)
    facility_data["population_imputed"] = facility_data["population"].isna() | (facility_data["population"] <= 0)
    facility_data["density_imputed"] = facility_data["density"].isna() | (facility_data["density"] <= 0)

    valid_sqft = facility_data.loc[facility_data["interior_sq_ft"] > 0]
    type_sqft_medians = valid_sqft.groupby("facility_type_name")["interior_sq_ft"].median()
    global_sqft_median = valid_sqft["interior_sq_ft"].median()
    if pd.isna(global_sqft_median) or global_sqft_median <= 0:
        global_sqft_median = 10000

    for facility_type, median_value in type_sqft_medians.items():
        mask = facility_data["interior_sq_ft_imputed"] & (facility_data["facility_type_name"] == facility_type)
        facility_data.loc[mask, "interior_sq_ft"] = median_value
    facility_data.loc[facility_data["interior_sq_ft_imputed"] & facility_data["interior_sq_ft"].isna(), "interior_sq_ft"] = global_sqft_median

    population_median = zip_data.loc[zip_data["population"] > 0, "population"].median()
    density_median = zip_data.loc[zip_data["density"] > 0, "density"].median()
    facility_data.loc[facility_data["population_imputed"], "population"] = population_median
    facility_data.loc[facility_data["density_imputed"], "density"] = density_median

    sqft_median = safe_log_median(facility_data["interior_sq_ft"])
    population_log_median = safe_log_median(facility_data["population"])
    density_log_median = safe_log_median(facility_data["density"])

    facility_data["sqft_score"] = np.log1p(facility_data["interior_sq_ft"]) / sqft_median
    facility_data["population_score"] = np.log1p(facility_data["population"]) / population_log_median
    facility_data["density_score"] = np.log1p(facility_data["density"]) / density_log_median
    facility_data["facility_type_multiplier"] = facility_data["facility_type_name"].map(FACILITY_TYPE_MULTIPLIERS).fillna(1.0)
    facility_data["facility_minimum_employees"] = facility_data["facility_type_name"].map(FACILITY_MINIMUMS).fillna(3).astype(int)

    facility_data["facility_staffing_weight"] = (
        facility_data["facility_type_multiplier"]
        * facility_data["sqft_score"].pow(0.50)
        * facility_data["population_score"].pow(0.30)
        * facility_data["density_score"].pow(0.20)
    )
    facility_data["facility_staffing_weight"] = facility_data["facility_staffing_weight"].replace([np.inf, -np.inf], np.nan).fillna(1.0)
    return facility_data


def allocate_largest_remainder(keys: pd.Series, weights: pd.Series, total: int) -> pd.Series:
    if total < 0:
        raise ValueError("Cannot allocate a negative total.")
    keys = keys.reset_index(drop=True)
    weights = pd.to_numeric(weights, errors="coerce").fillna(0).clip(lower=0).reset_index(drop=True)
    if len(keys) == 0:
        return pd.Series(dtype=int)
    if total == 0:
        return pd.Series(0, index=keys, dtype=int)
    if weights.sum() <= 0:
        weights = pd.Series(1.0, index=weights.index)

    raw = weights / weights.sum() * total
    base = np.floor(raw).astype(int)
    remainder_count = int(total - base.sum())
    remainders = raw - base
    order = remainders.sort_values(ascending=False, kind="mergesort").index[:remainder_count]
    base.loc[order] += 1
    return pd.Series(base.to_numpy(), index=keys, dtype=int)


def allocate_facility_targets(facility_data: pd.DataFrame) -> pd.DataFrame:
    targets = facility_data.copy()
    minimum_total = int(targets["facility_minimum_employees"].sum())
    if minimum_total > TOTAL_TARGET_EMPLOYEES:
        raise ValueError(
            f"Minimum staffing requires {minimum_total:,} employees, which exceeds TOTAL_TARGET_EMPLOYEES={TOTAL_TARGET_EMPLOYEES:,}."
        )

    remaining = TOTAL_TARGET_EMPLOYEES - minimum_total
    extra = allocate_largest_remainder(targets["facility_id"], targets["facility_staffing_weight"], remaining)
    targets["facility_target_employees"] = targets["facility_minimum_employees"] + targets["facility_id"].map(extra).astype(int)
    return targets


def allocate_department_targets(facility_targets: pd.DataFrame, departments: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    department_rows = []
    warning_rows = []
    facility_lookup = facility_targets.set_index("facility_id")

    for facility_id, group in departments.groupby("facility_id", sort=True):
        if facility_id not in facility_lookup.index:
            warning_rows.append({"facility_id": facility_id, "warning_type": "missing_facility", "details": "Department facility_id was not found in facility targets."})
            continue

        facility = facility_lookup.loc[facility_id]
        facility_type = facility["facility_type_name"]
        expected_shares = DEPARTMENT_SHARES.get(facility_type, {})
        expected_types = set(expected_shares)
        present_types = set(group["department_type_name"].dropna())

        for missing_type in sorted(expected_types - present_types):
            warning_rows.append({"facility_id": facility_id, "warning_type": "missing_department_type", "details": f"Expected {missing_type} for {facility_type}."})
        for unexpected_type in sorted(present_types - expected_types):
            warning_rows.append({"facility_id": facility_id, "warning_type": "unexpected_department_type", "details": f"Found {unexpected_type} for {facility_type}."})

        dept_count = len(group)
        target = int(facility["facility_target_employees"])
        if dept_count > target:
            raise ValueError(f"Facility {facility_id} has {dept_count} departments but only {target} target employees.")

        shares = group["department_type_name"].map(expected_shares).fillna(0.0).astype(float)
        if shares.sum() <= 0:
            shares = pd.Series(1.0, index=group.index)
        remaining = target - dept_count
        extras = allocate_largest_remainder(group["department_id"], shares, remaining)

        enriched = group.copy()
        enriched["department_share"] = shares / shares.sum()
        enriched["department_target_employees"] = 1 + enriched["department_id"].map(extras).astype(int)
        for column in facility_targets.columns:
            if column not in enriched.columns:
                enriched[column] = facility[column]
        department_rows.append(enriched)

    department_plan = pd.concat(department_rows, ignore_index=True) if department_rows else pd.DataFrame()
    warnings = pd.DataFrame(warning_rows)
    return department_plan, warnings


def choose_facility_manager_department(group: pd.DataFrame) -> int:
    facility_type = group["facility_type_name"].iloc[0]
    for department_type in FACILITY_MANAGER_PRIORITY.get(facility_type, []):
        candidates = group.loc[group["department_type_name"] == department_type, "department_id"]
        if not candidates.empty:
            return int(candidates.iloc[0])
    return int(group.sort_values("department_target_employees", ascending=False)["department_id"].iloc[0])


def generate_phone_number(rng: np.random.Generator) -> str:
    return f"555-{int(rng.integers(200, 999))}-{int(rng.integers(1000, 9999))}"


def generate_salary(rng: np.random.Generator, department_type: str, is_manager: bool, is_facility_manager: bool, manager_salary: float | None = None) -> float:
    if is_manager:
        low, high = MANAGER_SALARY_BANDS[department_type]
        if is_facility_manager:
            low = int(low + 0.75 * (high - low))
            high = int(high * 1.05)
        return float(int(rng.integers(low, high + 1) / 100) * 100)

    low, high = REGULAR_SALARY_BANDS[department_type]
    if manager_salary is not None:
        high = min(high, int(manager_salary) - 500)
    if high < low:
        low = max(0, high - 5000)
    return float(int(rng.integers(low, high + 1) / 100) * 100)


def generate_employees(department_plan: pd.DataFrame) -> pd.DataFrame:
    fake = Faker("en_US")
    Faker.seed(RANDOM_SEED)
    rng = np.random.default_rng(RANDOM_SEED)
    employees = []
    used_emails = set()

    facility_manager_departments = {
        facility_id: choose_facility_manager_department(group)
        for facility_id, group in department_plan.groupby("facility_id")
    }

    for _, dept in department_plan.sort_values(["facility_id", "department_id"]).iterrows():
        department_type = dept["department_type_name"]
        department_id = int(dept["department_id"])
        facility_id = int(dept["facility_id"])
        target = int(dept["department_target_employees"])
        is_facility_manager_department = facility_manager_departments[facility_id] == department_id

        manager_first = fake.first_name()
        manager_last = fake.last_name()
        manager_key = f"EMP_{facility_id}_{department_id}_0001"
        manager_salary = generate_salary(rng, department_type, True, is_facility_manager_department)
        manager_email = f"{slugify(manager_first)}.{slugify(manager_last)}.{facility_id}.{department_id}.1@{EMAIL_DOMAIN}"
        used_emails.add(manager_email)

        employees.append({
            "employee_import_key": manager_key,
            "facility_id": facility_id,
            "facility_name": dept["facility_name"],
            "facility_type_name": dept["facility_type_name"],
            "department_id": department_id,
            "department_name": dept["department_name"],
            "department_type_id": int(dept["department_type_id"]),
            "department_type_name": department_type,
            "is_department_manager": True,
            "is_facility_manager": bool(is_facility_manager_department),
            "manager_import_key": pd.NA,
            "full_name": truncate_text(f"{manager_first} {manager_last}", 50),
            "first_name": manager_first,
            "last_name": manager_last,
            "phone_number": generate_phone_number(rng),
            "email": truncate_text(manager_email, 100),
            "street_address": truncate_text(fake.street_address(), 100),
            "job_title": MANAGER_TITLES[department_type],
            "salary": manager_salary,
            "hours_worked": int(rng.integers(1850, 2201)),
            "employment_status": "Full-Time",
            "generated_source": GENERATED_SOURCE,
        })

        for sequence in range(2, target + 1):
            first_name = fake.first_name()
            last_name = fake.last_name()
            base_email = f"{slugify(first_name)}.{slugify(last_name)}.{facility_id}.{department_id}.{sequence}@{EMAIL_DOMAIN}"
            email = base_email
            duplicate_counter = 2
            while email in used_emails:
                email = f"{slugify(first_name)}.{slugify(last_name)}.{facility_id}.{department_id}.{sequence}.{duplicate_counter}@{EMAIL_DOMAIN}"
                duplicate_counter += 1
            used_emails.add(email)

            is_full_time = rng.random() < 0.85
            employees.append({
                "employee_import_key": f"EMP_{facility_id}_{department_id}_{sequence:04d}",
                "facility_id": facility_id,
                "facility_name": dept["facility_name"],
                "facility_type_name": dept["facility_type_name"],
                "department_id": department_id,
                "department_name": dept["department_name"],
                "department_type_id": int(dept["department_type_id"]),
                "department_type_name": department_type,
                "is_department_manager": False,
                "is_facility_manager": False,
                "manager_import_key": manager_key,
                "full_name": truncate_text(f"{first_name} {last_name}", 50),
                "first_name": first_name,
                "last_name": last_name,
                "phone_number": generate_phone_number(rng),
                "email": truncate_text(email, 100),
                "street_address": truncate_text(fake.street_address(), 100),
                "job_title": rng.choice(JOB_TITLES[department_type]),
                "salary": generate_salary(rng, department_type, False, False, manager_salary),
                "hours_worked": int(rng.integers(1850, 2201) if is_full_time else rng.integers(900, 1501)),
                "employment_status": "Full-Time" if is_full_time else "Part-Time",
                "generated_source": GENERATED_SOURCE,
            })

    return pd.DataFrame(employees)


def add_validation(report: list[dict[str, str]], check_name: str, passed: bool, details: str) -> None:
    report.append({"check_name": check_name, "status": "PASS" if passed else "FAIL", "details": details})


def run_validations(employee_import: pd.DataFrame, department_plan: pd.DataFrame, departments: pd.DataFrame, facility_targets: pd.DataFrame, warnings: pd.DataFrame) -> pd.DataFrame:
    report = []
    add_validation(report, "employee_import row count equals TOTAL_TARGET_EMPLOYEES", len(employee_import) == TOTAL_TARGET_EMPLOYEES, f"rows={len(employee_import):,}, target={TOTAL_TARGET_EMPLOYEES:,}")
    add_validation(report, "Every employee has a department_id", employee_import["department_id"].notna().all(), f"missing={employee_import['department_id'].isna().sum()}")
    add_validation(report, "Every department_id exists in departments.csv", set(employee_import["department_id"]).issubset(set(departments["department_id"])), "employee department IDs were compared against the export")

    dept_counts = employee_import.groupby("department_id").size()
    missing_employee_depts = set(departments["department_id"].dropna().astype(int)) - set(dept_counts.index.astype(int))
    add_validation(report, "Every department has at least one employee", len(missing_employee_depts) == 0, f"departments_without_employees={len(missing_employee_depts)}")

    manager_counts = employee_import.groupby("department_id")["is_department_manager"].sum()
    bad_manager_departments = manager_counts[manager_counts != 1]
    add_validation(report, "Every department has exactly one department manager", bad_manager_departments.empty, f"bad_departments={len(bad_manager_departments)}")

    facility_manager_counts = employee_import.groupby("facility_id")["is_facility_manager"].sum()
    bad_facility_managers = facility_manager_counts[facility_manager_counts != 1]
    add_validation(report, "Every facility has exactly one facility manager", bad_facility_managers.empty, f"bad_facilities={len(bad_facility_managers)}")

    regulars = employee_import[~employee_import["is_department_manager"]]
    add_validation(report, "Every regular employee has a manager_import_key", regulars["manager_import_key"].notna().all(), f"missing={regulars['manager_import_key'].isna().sum()}")
    manager_keys = set(employee_import["employee_import_key"])
    referenced_keys = set(regulars["manager_import_key"].dropna())
    add_validation(report, "Every manager_import_key exists as an employee_import_key", referenced_keys.issubset(manager_keys), f"missing_references={len(referenced_keys - manager_keys)}")
    add_validation(report, "Every email is unique", employee_import["email"].is_unique, f"duplicate_emails={employee_import['email'].duplicated().sum()}")

    salary_lookup = employee_import.set_index("employee_import_key")["salary"]
    regular_salary_check = regulars["salary"].to_numpy() <= regulars["manager_import_key"].map(salary_lookup).to_numpy()
    add_validation(report, "No regular employee salary exceeds assigned manager salary", bool(regular_salary_check.all()), f"violations={(~regular_salary_check).sum()}")

    dept_sum = department_plan.groupby("facility_id")["department_target_employees"].sum()
    facility_target_lookup = facility_targets.set_index("facility_id")["facility_target_employees"]
    aligned_facility_targets = dept_sum.index.to_series().map(facility_target_lookup)
    add_validation(report, "Department targets sum to facility targets", (dept_sum.to_numpy() == aligned_facility_targets.to_numpy()).all(), "department target totals were checked per facility")
    add_validation(report, "Facility targets sum to TOTAL_TARGET_EMPLOYEES", int(facility_targets["facility_target_employees"].sum()) == TOTAL_TARGET_EMPLOYEES, f"sum={int(facility_targets['facility_target_employees'].sum()):,}")

    below_min = facility_targets[facility_targets["facility_target_employees"] < facility_targets["facility_minimum_employees"]]
    add_validation(report, "No facility is below its minimum staffing", below_min.empty, f"below_minimum={len(below_min)}")

    unexpected = warnings[warnings.get("warning_type", pd.Series(dtype=str)).eq("unexpected_department_type")] if not warnings.empty else pd.DataFrame()
    add_validation(report, "No unexpected department type is silently ignored", True, f"unexpected_department_type_warnings_recorded={len(unexpected)}")

    length_limits = {"full_name": 50, "phone_number": 15, "email": 100, "street_address": 100, "job_title": 50}
    length_violations = []
    for column, limit in length_limits.items():
        count = (employee_import[column].astype(str).str.len() > limit).sum()
        if count:
            length_violations.append(f"{column}:{count}")
    add_validation(report, "No generated string exceeds likely schema limits", not length_violations, ", ".join(length_violations) if length_violations else "no violations")

    return pd.DataFrame(report)


def export_outputs(department_plan: pd.DataFrame, employee_import: pd.DataFrame, validation_report: pd.DataFrame) -> None:
    OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

    employee_export = employee_import.copy()
    flag_columns = ["is_department_manager", "is_facility_manager"]
    employee_export[flag_columns] = employee_export[flag_columns].astype(int)

    department_plan.to_csv(OUTPUT_PATH / "department_staffing_plan.csv", index=False)
    employee_export.to_csv(OUTPUT_PATH / "employee_import.csv", index=False)
    validation_report.to_csv(OUTPUT_PATH / "validation_report.csv", index=False)

    print(f"Exported files to: {OUTPUT_PATH.resolve()}")

## Load Inputs

In [4]:
facilities, zip_geo, departments = load_inputs()

display(facilities.head())
display(zip_geo.head())
display(departments.head())

,facility_name,facility_type_id,facility_type_name,county,street_address,city,state_code,zip_code,interior_sq_ft,territory_id,facility_name_key,facility_id
0,ABILENE - SOUTHERN HILLS STA,1,Post Office,TAYLOR,2501 BUFFALO GAP RD,ABILENE,TX,79605,17130.0,<NA>,ABILENE - SOUTHERN HILLS STA,9
1,ABILENE - MAIN OFFICE,1,Post Office,TAYLOR,341 PINE ST,ABILENE,TX,79601,122503.0,<NA>,ABILENE - MAIN OFFICE,10
2,ABILENE - AUXILIARY VMF,2,Vehicle Maintenance,TAYLOR,810 N 4TH ST,ABILENE,TX,79601,4370.0,<NA>,ABILENE - AUXILIARY VMF,11
3,ADDISON - MAIN OFFICE,1,Post Office,DALLAS,4900 AIRPORT PKWY,ADDISON,TX,75001,18564.0,<NA>,ADDISON - MAIN OFFICE,12
4,ALAMO - MAIN OFFICE,1,Post Office,HIDALGO,423 LOS ALAMOS DR,ALAMO,TX,78516,5049.0,<NA>,ALAMO - MAIN OFFICE,13


,zip_code,latitude,longitude,city,state,county_geoid,county_name,population,density,centroid_source
0,73960,36.49167,-101.79265,TEXHOMA,TX,48421,Sherman,5,3.2,uszips
1,75001,32.96015,-96.83808,ADDISON,TX,48113,Dallas,16872,1736.0,uszips
2,75002,33.08946,-96.60639,ALLEN,TX,48085,Collin,75057,771.1,uszips
3,75006,32.96165,-96.89717,CARROLLTON,TX,48113,Dallas,47807,1081.2,uszips
4,75007,33.00498,-96.89590,CARROLLTON,TX,48121,Denton,54646,1772.1,uszips


,department_name,department_type_id,manager_employee_id,manager_start_date,facility_id,department_id,created_at,updated_at,department_type_name
0,HOMDPO001_Delivery,2,1,2026-05-24 00:01:52,1,1,2026-05-23 23:56:14,2026-06-07 23:04:59,Delivery
1,HOMDPO001_Logistics,3,1,2026-05-24 00:01:52,1,2,2026-05-23 23:56:14,2026-06-07 23:04:59,Logistics
2,HAHOSHPO002_Retail Services,1,4,2026-05-26 12:56:05,2,3,2026-05-26 12:33:15,2026-06-07 23:10:37,Retail Services
3,HAHOSHPO002_Delivery,2,5,2026-05-26 12:56:05,2,4,2026-05-26 12:33:15,2026-06-07 23:04:59,Delivery
4,HAHOSHPO002_Logistics,3,3,2026-05-26 12:56:05,2,5,2026-05-26 12:33:15,2026-06-07 23:04:59,Logistics


## Department Export SQL Reference

This notebook uses the exported `departments.csv`. If you ever need to recreate it from MySQL, export the result of this query as CSV:

```sql
SELECT
    d.department_id,
    d.facility_id,
    d.department_name,
    d.department_type_id,
    dt.department_type_name
FROM departments d
JOIN department_type dt
    ON dt.department_type_id = d.department_type_id
ORDER BY d.facility_id, d.department_id;
```

## Facility Weights

In [5]:
facility_weights = calculate_facility_weights(facilities, zip_geo)
facility_targets = allocate_facility_targets(facility_weights)

weight_preview_columns = [
    "facility_id", "facility_name", "facility_type_name", "zip_code", "interior_sq_ft", "population", "density",
    "sqft_score", "population_score", "density_score", "facility_staffing_weight", "facility_target_employees",
]
display(facility_targets[weight_preview_columns].head())

,facility_id,facility_name,facility_type_name,zip_code,interior_sq_ft,population,density,sqft_score,population_score,density_score,facility_staffing_weight,facility_target_employees
0,9,ABILENE - SOUTHERN HILLS STA,Post Office,79605,17130.0,28467.0,1000.8,1.032121,1.015532,1.328580,1.080316,10
1,10,ABILENE - MAIN OFFICE,Post Office,79601,122503.0,28050.0,41.2,1.240401,1.014071,0.719599,1.047176,10
2,11,ABILENE - AUXILIARY VMF,Vehicle Maintenance,79601,4370.0,28050.0,41.2,0.887509,1.014071,0.719599,0.620045,6
3,12,ADDISON - MAIN OFFICE,Post Office,75001,18564.0,16872.0,1736.0,1.040632,0.963741,1.434405,1.084354,10
4,13,ALAMO - MAIN OFFICE,Post Office,78516,5049.0,34370.0,350.9,0.902797,1.034189,1.127414,0.983086,9


## Department Staffing Plan

In [6]:
department_plan, department_validation_warnings = allocate_department_targets(facility_targets, departments)

department_staffing_columns = [
    "facility_id", "facility_name", "facility_type_name", "zip_code", "interior_sq_ft", "population", "density",
    "interior_sq_ft_imputed", "population_imputed", "density_imputed", "sqft_score", "population_score", "density_score",
    "facility_type_multiplier", "facility_staffing_weight", "facility_minimum_employees", "facility_target_employees",
    "department_id", "department_name", "department_type_id", "department_type_name", "department_share", "department_target_employees",
]
department_plan = department_plan[department_staffing_columns]

display(department_plan.head())
display(department_validation_warnings.head())

,facility_id,facility_name,facility_type_name,zip_code,interior_sq_ft,population,density,interior_sq_ft_imputed,population_imputed,density_imputed,sqft_score,population_score,density_score,facility_type_multiplier,facility_staffing_weight,facility_minimum_employees,facility_target_employees,department_id,department_name,department_type_id,department_type_name,department_share,department_target_employees
0,1,Main Demo Post Office,Post Office,77001,12431.0,6149.5,32.4,True,True,True,0.998176,0.863819,0.674631,1.0,0.883779,4,9,1,HOMDPO001_Delivery,2,Delivery,0.55,4
1,1,Main Demo Post Office,Post Office,77001,12431.0,6149.5,32.4,True,True,True,0.998176,0.863819,0.674631,1.0,0.883779,4,9,2,HOMDPO001_Logistics,3,Logistics,0.20,2
2,1,Main Demo Post Office,Post Office,77001,12431.0,6149.5,32.4,True,True,True,0.998176,0.863819,0.674631,1.0,0.883779,4,9,1091,HOMDPO001_Retail Services,1,Retail Services,0.25,3
3,2,HOUSTON - MAIN OFFICE,Post Office,77002,25206.0,21015.0,4119.4,False,False,False,1.073013,0.985482,1.600496,1.0,1.133047,4,10,3,HAHOSHPO002_Retail Services,1,Retail Services,0.25,3
4,2,HOUSTON - MAIN OFFICE,Post Office,77002,25206.0,21015.0,4119.4,False,False,False,1.073013,0.985482,1.600496,1.0,1.133047,4,10,4,HAHOSHPO002_Delivery,2,Delivery,0.55,5


""


## Generate Employees

In [7]:
employee_import = generate_employees(department_plan)
display(employee_import.head())

,employee_import_key,facility_id,facility_name,facility_type_name,department_id,department_name,department_type_id,department_type_name,is_department_manager,is_facility_manager,manager_import_key,full_name,first_name,last_name,phone_number,email,street_address,job_title,salary,hours_worked,employment_status,generated_source
0,EMP_1_1_0001,1,Main Demo Post Office,Post Office,1,HOMDPO001_Delivery,2,Delivery,True,True,NaN,Danielle Johnson,Danielle,Johnson,555-818-6890,danielle.johnson.1.1.1@postal-demo.local,32181 Johnson Course Apt. 389,Delivery Manager,95400.0,2004,Full-Time,staffing_plan_employee_generation.ipynb
1,EMP_1_1_0002,1,Main Demo Post Office,Post Office,1,HOMDPO001_Delivery,2,Delivery,False,False,EMP_1_1_0001,Anthony Gonzalez,Anthony,Gonzalez,555-268-7275,anthony.gonzalez.1.1.2@postal-demo.local,79402 Peterson Drives Apt. 511,City Carrier,43800.0,1216,Part-Time,staffing_plan_employee_generation.ipynb
2,EMP_1_1_0003,1,Main Demo Post Office,Post Office,1,HOMDPO001_Delivery,2,Delivery,False,False,EMP_1_1_0001,Noah Howard,Noah,Howard,555-979-7456,noah.howard.1.1.3@postal-demo.local,407 Teresa Lane Apt. 849,Mail Carrier,52200.0,1894,Full-Time,staffing_plan_employee_generation.ipynb
3,EMP_1_1_0004,1,Main Demo Post Office,Post Office,1,HOMDPO001_Delivery,2,Delivery,False,False,EMP_1_1_0001,Nicole Ward,Nicole,Ward,555-599-4336,nicole.ward.1.1.4@postal-demo.local,103 Galloway Walk,City Carrier,60500.0,2124,Full-Time,staffing_plan_employee_generation.ipynb
4,EMP_1_2_0001,1,Main Demo Post Office,Post Office,2,HOMDPO001_Logistics,3,Logistics,True,False,NaN,Jesse Garcia,Jesse,Garcia,555-521-8404,jesse.garcia.1.2.1@postal-demo.local,5255 Elizabeth Squares Apt. 928,Logistics Manager,93400.0,2041,Full-Time,staffing_plan_employee_generation.ipynb


## Validation

In [8]:
validation_report = run_validations(employee_import, department_plan, departments, facility_targets, department_validation_warnings)
display(validation_report)

failed_checks = validation_report[validation_report["status"] == "FAIL"]
if failed_checks.empty:
    print("All validation checks passed.")
else:
    print("Some validation checks failed. Review validation_report before loading to MySQL.")

,check_name,status,details
0,employee_import row count equals TOTAL_TARGET_...,PASS,"rows=5,000, target=5,000"
1,Every employee has a department_id,PASS,missing=0
2,Every department_id exists in departments.csv,PASS,employee department IDs were compared against ...
3,Every department has at least one employee,PASS,departments_without_employees=0
4,Every department has exactly one department ma...,PASS,bad_departments=0
5,Every facility has exactly one facility manager,PASS,bad_facilities=0
6,Every regular employee has a manager_import_key,PASS,missing=0
7,Every manager_import_key exists as an employee...,PASS,missing_references=0
8,Every email is unique,PASS,duplicate_emails=0
9,No regular employee salary exceeds assigned ma...,PASS,violations=0


All validation checks passed.


## Summary Tables

In [9]:
employee_counts_by_facility_type = employee_import.groupby("facility_type_name").size().reset_index(name="employee_count").sort_values("employee_count", ascending=False)
employee_counts_by_department_type = employee_import.groupby("department_type_name").size().reset_index(name="employee_count").sort_values("employee_count", ascending=False)
top_20_facilities = facility_targets.sort_values("facility_target_employees", ascending=False).head(20)[["facility_id", "facility_name", "facility_type_name", "facility_target_employees"]]
bottom_20_facilities = facility_targets.sort_values("facility_target_employees", ascending=True).head(20)[["facility_id", "facility_name", "facility_type_name", "facility_target_employees"]]
imputation_counts = facility_targets[["interior_sq_ft_imputed", "population_imputed", "density_imputed"]].sum().reset_index()
imputation_counts.columns = ["field", "imputed_count"]
validation_summary = validation_report.groupby("status").size().reset_index(name="check_count")

display(employee_counts_by_facility_type)
display(employee_counts_by_department_type)
display(top_20_facilities)
display(bottom_20_facilities)
display(imputation_counts)
display(validation_summary)

,facility_type_name,employee_count
3,Post Office,4699
1,Mail Processing,146
4,Vehicle Maintenance,89
2,Network Facilities,39
0,Administrative Office,27


,department_type_name,employee_count
1,Delivery,2274
5,Retail Services,1417
2,Logistics,1074
3,Maintenance,114
4,Operations,74
0,Administrative,47


,facility_id,facility_name,facility_type_name,facility_target_employees
66,75,BRYAN - P&DC-MAIN OFFICE-SHF,Mail Processing,30
45,54,BEAUMONT - BEAUMONT TX LPC; VMF,Mail Processing,30
363,367,MCALLEN - AUXILIARY VMF; Main Office; P&DC,Mail Processing,30
539,5,USPS North Houston TX Distribution Center,Mail Processing,28
351,355,LUBBOCK - MAIN OFFICE/ P&DC; VMF,Mail Processing,28
540,6,South Houston Local Processing Center,Network Facilities,21
507,511,UVALDE - MAIN OFFICE,Network Facilities,18
155,164,DENTON - MAIN OFFICE,Post Office,11
274,280,HOUSTON - WILLIAM RICE,Post Office,11
7,16,ALVIN - MAIN OFFICE,Post Office,10


,facility_id,facility_name,facility_type_name,facility_target_employees
527,531,WICHITA FALLS - VMF,Vehicle Maintenance,6
494,498,SYLVESTER - MPO/MODULAR BLDG,Post Office,6
495,499,TALPA - MPO/MODULAR BLDG,Post Office,6
326,330,LASALLE - MO/MODULAR BLDG,Post Office,6
312,316,KILDARE - MPO/MODULAR BLDG,Post Office,6
361,365,MCADOO - MAIN OFFICE,Post Office,6
201,210,FRANCITAS - MPO/MODULAR BLDG,Post Office,6
47,56,BEDFORD - CENTRAL STATION,Administrative Office,6
219,228,GIRARD - MPO/MODULAR BLDG,Post Office,6
373,377,MILLERSVIEW - MPO/MODULAR BLDG,Post Office,6


,field,imputed_count
0,interior_sq_ft_imputed,8
1,population_imputed,14
2,density_imputed,14


,status,check_count
0,PASS,15


## Export CSV Files

In [10]:
export_outputs(department_plan, employee_import, validation_report)

Exported files to: Z:\Computer Science\GitHub Repositories\Personal Projects\Synthetic-Texas-Postal-Dataset\Synthetic MySQL Tables\Employee\output


## Later MySQL Loading Workflow

Use `employee_import.csv` as a staging input instead of inserting directly from Python.

1. Load `employee_import.csv` into a staging table that preserves `employee_import_key` and `manager_import_key`.
2. Insert department managers first, leaving `manager_employee_id` null.
3. Insert regular employees second by resolving `manager_import_key` to the manager row inserted from `employee_import_key`.
4. Update `departments.manager_employee_id` from the inserted department manager employees.
5. Update `facility.manager_employee_id` from the inserted facility manager employees.
6. Drop or archive the staging table after validation.